# Minimal *C. elegans* aggregation: two individual-based models

**Question:** Can local density-dependent slowing sustain aggregation, and what changes when the feedback acts through oxygen instead?

Every worm has a position and direction. Its body is a **visual trail only**. The two sections below share the same arena, random motion, initial conditions and measurements. First investigate accumulation; connected walls and holes are a later, optional question.

**Quick start:** select a Python kernel, run the setup and the live cell in section 1 or 2, then click **Start / resume**. Change a slider and click **Apply & reset** to start a new experiment. Pause one section before running the other. **Steps / frame** changes playback speed, not the numerical time step. Running all cells is safe: animations and expensive comparisons do not start automatically.

If imports fail, uncomment the install cell below, run it, and restart the kernel. Use JupyterLab, classic Notebook with widgets enabled, or a VS Code Jupyter notebook; a static preview cannot operate the controls. Keep this notebook beside `model.py`, `live.py`, and `experiments.py`.

### Scope and assumptions

- 2D periodic square, side 32 body lengths; no walls or pre-drawn pattern.
- 1,000 individual point agents by default. No explicit attraction, alignment or physical body collisions. Overlap is allowed; we cannot interpret dense packing as realistic worm packing.
- A heading persists for approximately 8 model time units; random turning changes it continuously. Free speed = 1 body length per time unit. These units are **not calibrated seconds**.
- Population density is deposited conservatively on a 64 × 64 grid and smoothed with a Gaussian of width 1 body length. This is a measurement of discrete agents, **not a density evolution equation**. It includes each agent's own small contribution. The same density measurement drives oxygen depletion.
- Tails have fixed spatial length, are resampled along the recent path, and never affect motion or oxygen. They do not leave marks behind.
- Positive minimum speed prevents permanently frozen agents. Apparent stability must still be checked over long runs: finite-time persistence is not proof of an asymptotically stationary pattern.

In [ ]:
# Only if dependencies are missing; restart the kernel afterwards.
# %pip install numpy scipy matplotlib ipywidgets ipykernel

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
from dataclasses import replace, asdict
import csv
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

# Supports opening from this folder or the repository root.
HERE = Path.cwd()
if not (HERE / "model.py").exists():
    HERE = HERE / "minimal_aggregation"
if not (HERE / "model.py").exists():
    raise RuntimeError("Open the notebook with its minimal_aggregation folder as the working directory.")
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
from model import Parameters, Simulation, random_reference, run_experiment
from live import LiveSimulation, plot_experiment
from experiments import comparison

# Shared defaults. Most experiments need only the sliders below.
COMMON = Parameters()
print(f"Ready: {COMMON.n} worms, {COMMON.grid} × {COMMON.grid} grid, dt={COMMON.dt}.")

## 1. Without oxygen — local crowding changes speed

Each time step, the heading receives zero-mean random angular noise and the worm moves in that direction:

$$
\theta_{t+\Delta t}=\theta_t+\sqrt{2\Delta t/\tau}\,\xi,
\qquad \mathbf x_{t+\Delta t}=\mathbf x_t+v\,\mathbf e(\theta_{t+\Delta t})\Delta t.
$$

Here $\xi$ is an independent standard normal draw, and $\tau$ is the persistence time. Speed depends on local population density:

$$
v(\rho)/v_0=\varepsilon+(1-\varepsilon)\exp[-a\rho/\rho_*].
$$

$a$ is the **Slowing** slider; $\rho_*=1$ worm per square body length is a fixed reference density. The floor $\varepsilon$ is listed with the advanced constants below. There is **no steering toward neighbors**. Aggregation, if it occurs, comes from longer residence in crowded regions.

**Try these in order:**

1. Keep the defaults and run to at least $t=500$, then beyond $t=2000$.
2. Set **Slowing = 0** and reset with the same seed. This is the constant-speed control.
3. Try Slowing = 0.5, 1.0 and 1.5 with the same worm count and persistence. Weak slowing may not sustain aggregation.
4. Repeat the comparison with seeds 17 and 29. Increasing slowing indefinitely can make formation slower, because almost everyone barely moves; it is not necessarily a stronger clustering experiment.

**Read the plots:** density CV is standard deviation / mean of the smoothed density. Random discrete points already have a nonzero CV (dashed reference). The second line is the fraction of smoothed mass in the largest connected region above **1.5 × global mean density**, accounting for periodic edges. It is a threshold-dependent morphology diagnostic, not an exact count of worms in a biological cluster. All density displays use the same smoothing scale.

In [ ]:
# Re-running this cell stops its previous animation.
if "without_oxygen" in globals():
    without_oxygen.close()
without_oxygen = LiveSimulation(replace(COMMON, mode="density"))

**Optional long run of the current settings.** This starts a fresh run with the applied settings, without rendering every frame. It does not continue or overwrite the live state. Keep sampling intervals equal when comparing density-map changes.

In [ ]:
RUN_DENSITY_LONG = False  # Change to True and run this cell when ready.
if RUN_DENSITY_LONG:
    without_oxygen.pause()
    density_parameters = without_oxygen.sim.p
    density_records, density_snapshots, density_final = run_experiment(
        density_parameters, duration=3000, sample_every=50,
        snapshot_times=(500, 1500, 3000))
    plot_experiment(density_records, density_snapshots, density_parameters)

## 2. With oxygen — local environmental feedback

There is **no direct density-dependent speed rule** in this section. Density changes oxygen, and oxygen changes motion. Oxygen starts spatially uniform at the ambient value, $o=1$.

Each grid cell exchanges oxygen with its four neighbors, replenishes toward the ambient value and loses oxygen locally:

$$
\dot o_j=D\,\Delta_h o_j+\lambda(1-o_j)-k\rho_j o_j.
$$

This is an environmental concentration field; the worms remain individual agents. The depletion term represents **worms plus associated bacteria**, not measured worm respiration alone. Bacteria are not separately simulated. The oxygen-dependent sink switches off at zero oxygen. Diffusion is advanced with the exact discrete-grid propagator and the local reaction is integrated exactly; splitting still has finite-step error.

Speed is smallest near a preferred oxygen value, and rises on either side:

$$
v(o)/v_0=\varepsilon+(1-\varepsilon)\left[1-\exp\left(-((o-o_*)/w)^2\right)\right].
$$

The heading also turns down the gradient of the mismatch potential $U=(o-o_*)^2/2$:

$$
\Delta\theta=-\chi(o-o_*)\,\mathbf e_\perp(\theta)\cdot\nabla_h o\,\Delta t
+\sqrt{2\Delta t/\tau}\,\xi.
$$

Above the preferred oxygen level, this favors lower oxygen; below it, the preference reverses. This is a coarse behavioral rule, not a neural model. $o_*=0.4$ is roughly 8.4% if ambient 1 is interpreted as 21%; that is a **motivation**, not a calibrated conversion for this environment. We use a generic oxygen-responsive worm, not a fitted N2 or npr-1 strain.

**Main sliders:** O₂ depletion = $k$; O₂ turning = $\chi$. Worm count, persistence and seed have exactly the same meaning as in section 1.

**Useful controls:**

1. Defaults: density → depletion → movement feedback.
2. **O₂ depletion = 0**: oxygen stays uniform at 1. No oxygen gradient or self-generated environmental feedback is possible.
3. **O₂ turning = 0**, with depletion retained: speed still depends on oxygen, but directed aerotaxis is removed. This separates residence-time effects from turning.
4. Compare both models with the **same worm count, persistence and seed**. This is a comparison of two mechanisms, not evidence that oxygen is necessary in the real animal.

**Stability:** inspect late snapshots as well as the curves. A stable CV can coexist with merging or drifting clusters. The minimum speed is positive, so worms can keep exchanging with a persistent aggregate. Long-run batch results below check multiple seeds.

In [ ]:
if "with_oxygen" in globals():
    with_oxygen.close()
with_oxygen = LiveSimulation(replace(COMMON, mode="oxygen"))

**Optional long run of the applied oxygen settings.**

In [ ]:
RUN_OXYGEN_LONG = False
if RUN_OXYGEN_LONG:
    with_oxygen.pause()
    oxygen_parameters = with_oxygen.sim.p
    oxygen_records, oxygen_snapshots, oxygen_final = run_experiment(
        oxygen_parameters, duration=3000, sample_every=50,
        snapshot_times=(500, 1500, 3000))
    plot_experiment(oxygen_records, oxygen_snapshots, oxygen_parameters)

### Long-run comparison and saved reference results

The supplied reference run uses **three seeds (4, 17, 29), 3,000 time units, and the shared defaults**, not the current slider positions. This is 375 orientation-persistence times. The first panel shows one seed; the metric plot shows all three. Numerical data and the exact parameters are saved in `results/`.

The summary reports mean CV and largest-region mass over the final third. Map change measures movement/rearrangement over 50 time units, normalized by the previous density map's norm. Half-to-end correlation tests whether the spatial arrangement remains similar from t=1500 to t=3000; drift also reduces it. No universal threshold here proves stability.

Use the switch below to regenerate a **separate** comparison with `COMMON` (not individually adjusted sliders). It pauses both animations. For a fair test, modify `COMMON` once and keep it the same for both models.

In [ ]:
RUN_COMPARISON = False
if RUN_COMPARISON:
    if "without_oxygen" in globals(): without_oxygen.pause()
    if "with_oxygen" in globals(): with_oxygen.pause()
    comparison(HERE / "results_custom", duration=3000, seeds=(4, 17, 29), base=COMMON)

# Switch to results_custom after running your own comparison.
RESULTS = HERE / "results"
if (RESULTS / "summary.csv").exists():
    with (RESULTS / "summary.csv").open() as stream:
        summary = list(csv.DictReader(stream))
    table = "| Model | Seed | Random CV | Late CV | Largest region | Map correlation, half → end |\n"
    table += "|---|---:|---:|---:|---:|---:|\n"
    for row in summary:
        table += (f"| {row['mode']} | {row['seed']} | {float(row['random_cv']):.2f} | "
                  f"{float(row['late_cv_mean']):.2f} | {float(row['late_largest_mean']):.0%} | "
                  f"{float(row['half_to_end_map_correlation']):.2f} |\n")
    display(Markdown(table))
    display(Image(filename=str(RESULTS / "comparison.png")))
    display(Image(filename=str(RESULTS / "long_run_metrics.png")))
else:
    print("No saved comparison yet. Set RUN_COMPARISON = True to generate one.")

### Findings from the supplied runs

Both models sustain accumulation across all three seeds. The density-only model retains several clusters but continues slowly coarsening/rearranging; late CV is 1.55–1.60. The oxygen model forms a persistent ring-like aggregate; late CV is about 2.30 and its largest dense region contains about 96% of smoothed mass. Random placement gives CV about 0.28. Neither default gives the many-cell network in the reference photograph.

The shared minimum speed is **1% of free speed**, a strong retention assumption. This is not a biologically fitted value. Test its influence with `replace(COMMON, min_speed=0.04)` and rerun both models. Positive speed prevents absolute freezing, but very slow escape can still make patterns long-lived.

Additional checks used three seeds and 1,000 time units per run. Without slowing, CV remained 0.277–0.284; without oxygen depletion it remained 0.276–0.286. With oxygen-dependent speed but **no directed turning**, CV was 0.275–0.306: the chosen oxygen parameters require turning for strong aggregation within this test. This is a parameter-specific result, not a universal necessity claim.

Halving dt and, separately, doubling grid resolution changed mean late CV by less than 2%. Individual shapes and largest-region masses vary more. Eight numerical/behavioral tests and notebook execution passed, including the Start, Pause, Advance and Reset callbacks. The rendered plots were inspected; widget interaction was exercised through the kernel, not manual clicks in every notebook frontend.

### What would count as the next result?

- **Sustained accumulation:** CV stays clearly above the random-placement reference across seeds and late times. Inspect snapshots to distinguish many groups from one compact aggregate.
- **Relatively stable patterns:** late morphology persists and map changes are modest; report continued coarsening, drift and fluctuations rather than calling them stationary.
- **Cell-like networks:** connected bands enclose multiple low-density regions and persist. A single oxygen ring or a few blobs does not meet this target. Visual bodies must not be mistaken for persistent network walls.

To explore morphology after establishing aggregation, vary worm number at fixed arena size, then compare persistence or oxygen depletion one at a time. Reuse the same seeds. Do not add adhesion or alignment just to force a desired image. If the minimal models coarsen to a single aggregate, that is a valid result and a reason to formulate a new question about what could maintain a network.

### Advanced constants, numerical checks and limits

The next cell shows every parameter. Keeping these visible prevents the simple interface from hiding assumptions. The interface exposes only the most useful experimental controls; you can create a new `Parameters` instance or use `replace(COMMON, ...)` for a deliberate advanced experiment.

Time step = 0.1, sensing width = 1, density reference = 1, oxygen diffusion = 1, replenishment = 0.08, preferred oxygen = 0.4, response width = 0.25. The replenishment/diffusion length scale is about 3.5 body lengths. These are model choices, not fitted measurements. Oxygen sensing and density slowing use different response curves; this is **not** a speed-matched causal inference experiment.

For a numerical check, rerun representative settings with `dt=0.05` and separately `grid=128`, keeping `sensing=1` (physical width) fixed. Compare multi-seed statistics, not exact trajectories: changing dt changes the sequence of random increments. Extremely strong steering may require a smaller dt. The oxygen integration preserves positivity up to round-off, but that does not eliminate splitting or agent-step errors.

`test_model.py` checks mass conservation, periodicity, oxygen bounds and equilibrium, control behavior, random-seed reproducibility and that turning off trails leaves agent dynamics unchanged. `results/validation.json`, if present, records the additional controls and resolution checks used for the delivered defaults.

The point-agent approximation permits unrealistically high local packing. The grid/smoothing scale sets how narrow a resolved structure can be. Missing body mechanics, bacteria redistribution, food depletion and strain-specific measurements limit biological interpretation. The model tests sufficiency of its rules within the tested settings, not the true mechanism in an experiment.

### Biological motivation

- [Demir et al. (2020), Dynamics of pattern formation and emergence of swarming in C. elegans](https://doi.org/10.7554/eLife.52781): motivates density-associated oxygen depletion, replenishment, oxygen-sensitive motility and patterns. The present agent model and response functions are independent simplifications, not a reproduction of their continuum equations.
- [Gray et al. (2004), Oxygen sensation and social feeding](https://doi.org/10.1038/nature02714): motivates intermediate oxygen preference; responses depend on biological context.
- [Ding et al. (2019), Shared behavioral mechanisms underlie C. elegans aggregation and swarming](https://doi.org/10.7554/eLife.43318): agent-based precedent, including additional reversal and taxis rules absent from the density-only baseline here.

In [ ]:
asdict(COMMON)